In [1]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)
API_TOKEN = os.getenv("TMDB_READ_TOKEN").strip("'\" ")

In [10]:
def fetch_popular_movies(pages=5):
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {API_TOKEN}"
    }
    
    movies = []
    for page in range(1, pages + 1):
        url = f"https://api.themoviedb.org/3/movie/popular?language=en-US&page={page}"
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            movies.extend(response.json().get('results', []))
        else:
            print(f"Failed page {page}: Status {response.status_code} - {response.text}")
            
    return pd.DataFrame(movies)

# 3. Fetch data & slice required columns
df = fetch_popular_movies(pages=6)

if not df.empty:
    df = df[['id', 'title', 'overview', 'vote_average', 'popularity']].dropna(subset=['overview'])
    print(f" Successfully loaded {len(df)} movies into DataFrame!\n")
    display(df.head())
else:
    print("❌ Failed to fetch movies.")

 Successfully loaded 120 movies into DataFrame!



,id,title,overview,vote_average,popularity
0,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",7.900,937.0796
1,1108427,Moana,"Teenage Moana answers the Ocean's call and, fo...",5.765,607.7654
2,1275779,Disclosure Day,A cybersecurity expert becomes a whistleblower...,7.000,559.6616
3,1339713,Obsession,"After breaking the mysterious ""One Wish Willow...",8.245,490.5794
4,1083381,Backrooms,A strange doorway appears in the basement of a...,7.100,445.1714


In [4]:
# Double check for empty overviews
df['overview'] = df['overview'].fillna('')

# Reset index so indices in df cleanly align with the similarity matrix
df = df.reset_index(drop=True)

print(f"Total movies ready for vectorization: {len(df)}")

Total movies ready for vectorization: 100


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Construct the TF-IDF matrix (rows = movies, columns = unique words)
tfidf_matrix = tfidf.fit_transform(df['overview'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Extracted {tfidf_matrix.shape[1]} unique word features across {tfidf_matrix.shape[0]} movies.")

TF-IDF Matrix Shape: (100, 1573)
Extracted 1573 unique word features across 100 movies.


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")

Cosine Similarity Matrix Shape: (100, 100)


In [7]:
import numpy as np

# Create reverse lookup mapping movie titles (lowercase) to row index
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim, top_n=5):
    clean_title = title.lower().strip()
    
    # Handle missing title edge-case
    if clean_title not in indices:
        return f"Movie '{title}' not found in current dataset."
    
    # Get index of input movie
    idx = indices[clean_title]
    
    # Extract similarity scores for all movies with the input movie
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort movies by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top_n recommendations (exclude index 0 since that is the movie itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # Collect indices of recommended movies
    movie_indices = [i[0] for i in sim_scores]
    
    # Format output DataFrame
    results = df.iloc[movie_indices][['title', 'vote_average', 'overview']].copy()
    results['similarity_score'] = [np.round(i[1], 3) for i in sim_scores]
    
    return results

In [8]:
# Grab the title of the first movie in your dataset
sample_title = df['title'].iloc[1]
print(f"Testing recommendations for: '{sample_title}'\n")

# Run recommendation engine
get_recommendations(sample_title, top_n=5)

Testing recommendations for: 'Moana'



,title,vote_average,overview,similarity_score
72,Moana 2,6.995,After receiving an unexpected call from her wa...,0.109
53,Shelter,7.764,A man living in self-imposed exile on a remote...,0.046
55,23 000 Lives,5.189,A group of young people sets sail for the Medi...,0.044
27,The Super Mario Galaxy Movie,8.251,Having thwarted Bowser's previous plot to marr...,0.040
97,Predator: Badlands,7.822,"Cast out from his clan, a young Predator finds...",0.038


In [9]:
import pickle

# 1. Export the cleaned DataFrame (convert to dictionary or raw df)
pickle.dump(df.to_dict(orient='records'), open('movies.pkl', 'wb'))

# 2. Export the pre-computed Cosine Similarity Matrix
pickle.dump(cosine_sim, open('similarity.pkl', 'wb'))

print(" Successfully exported 'movies.pkl' and 'similarity.pkl'!")

 Successfully exported 'movies.pkl' and 'similarity.pkl'!
